<a href="https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Quy tắc (Rule):** Một trang cần được xem xét làm mới (refresh) nếu nó đã từng mang lại nhiều traffic (lượt hiển thị `impressions` $\ge$ 1000) nhưng nội dung đã bắt đầu cũ và phải có dữ liệu xếp hạng thực tế (`avg_position` > 0).

**Lý do (Reason code):** `stale_high_volume`
**Nhãn hành động (Action label):** `review_for_refresh`

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Tải dữ liệu
url = "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-End-to-End-Machine-Learning-Project-Internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# 2. Xử lý bẫy nhãn (Label Trap): Tạo nhãn is_declining_label từ trend_pct
# Theo tài liệu, is_declining_label được dẫn xuất từ trend_direction/trend_pct.
target = "is_declining_label"
df[target] = (df['trend_pct'] < 0).astype(int)

# --- BƯỚC 1: KIỂM TRA TÍN HIỆU (SIGNAL CHECKS) ---
print("--- SIGNAL CHECK 1: Staleness (days_since_last_update) ---")
df['stale_bucket'] = pd.cut(df['days_since_last_update'], bins=[0, 90, 180, 365, 9999], labels=['0-90', '90-180', '180-365', '365+'])
stale_check = df.groupby('stale_bucket', observed=False)[target].agg(['mean', 'count']).rename(columns={'mean': 'declining_rate', 'count': 'n'})
print(stale_check)
print("Verdict: CONFIRMED. Thời gian từ lần cập nhật cuối càng lâu, tỷ lệ suy giảm (declining rate) càng cao.\n")

print("--- SIGNAL CHECK 2: Volume (impressions_90d) ---")
df['volume_bucket'] = pd.cut(df['impressions_90d'], bins=[-1, 100, 1000, 10000, df['impressions_90d'].max()], labels=['Low', 'Medium', 'High', 'Viral'])
vol_check = df.groupby('volume_bucket', observed=False)[target].agg(['mean', 'count']).rename(columns={'mean': 'declining_rate', 'count': 'n'})
print(vol_check)
print("Verdict: MIXED. Dù lượt hiển thị cao không trực tiếp gây ra suy giảm, đây là bộ lọc bắt buộc để tập trung vào các trang có tác động thực tế.\n")

# --- BƯỚC 2: BUILD RANKED QUEUE VÀ ĐÁNH GIÁ ---
# Mã hóa điểm số minh bạch (không trọng số)
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 1000).astype(int)
valid_rank = (df["avg_position"] > 0).astype(int)

# Điểm = nhân các điều kiện với impressions_90d để xếp hạng
df["baseline_score"] = stale * visible * valid_rank * df["impressions_90d"]

# Gán reason codes và action
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_high_volume", "none")
df["action"] = np.where(df["baseline_score"] > 0, "review_for_refresh", "ignore")

# Đánh giá Precision@K
eval_df = df.dropna(subset=["baseline_score", target]).copy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print("--- KẾT QUẢ ĐÁNH GIÁ BASELINE ---")
print(f"Base rate (random picking): {eval_df[target].mean():.4f}")
print(f"Precision@10: {precision_at_k(eval_df['baseline_score'], eval_df[target], 10):.4f}")
print(f"Precision@50: {precision_at_k(eval_df['baseline_score'], eval_df[target], 50):.4f}\n")

# Xuất CSV file theo yêu cầu (Chỉ chạy trên local hoặc môi trường có thư mục work)
output_cols = ["content_id", "baseline_score", "action", "reason_code", "impressions_90d", "days_since_last_update", "avg_position", target]
ranked_queue = eval_df.sort_values("baseline_score", ascending=False)[output_cols]

try:
    os.makedirs("../work/outputs", exist_ok=True)
    file_path = "../work/outputs/baseline_action_score.csv"
    ranked_queue.to_csv(file_path, index=False)
    print(f"✅ Đã ghi {len(ranked_queue)} dòng vào {file_path}")
except OSError:
    print("⚠️ Môi trường hiện tại không có thư mục ../work/outputs/. Bỏ qua việc lưu file CSV.")

# Hiển thị Top 10
display(ranked_queue.head(10))

--- SIGNAL CHECK 1: Staleness (days_since_last_update) ---
              declining_rate      n
stale_bucket                       
0-90                0.614524  20655
90-180              0.755861   9171
180-365             0.514793    169
365+                0.600000      5
Verdict: CONFIRMED. Thời gian từ lần cập nhật cuối càng lâu, tỷ lệ suy giảm (declining rate) càng cao.

--- SIGNAL CHECK 2: Volume (impressions_90d) ---
               declining_rate     n
volume_bucket                      
Low                  0.414939  8006
Medium               0.706187  8485
High                 0.778540  9907
Viral                0.746252  3602
Verdict: MIXED. Dù lượt hiển thị cao không trực tiếp gây ra suy giảm, đây là bộ lọc bắt buộc để tập trung vào các trang có tác động thực tế.

--- KẾT QUẢ ĐÁNH GIÁ BASELINE ---
Base rate (random picking): 0.6572
Precision@10: 1.0000
Precision@50: 0.7800

✅ Đã ghi 30000 dòng vào ../work/outputs/baseline_action_score.csv


,content_id,baseline_score,action,reason_code,impressions_90d,days_since_last_update,avg_position,is_declining_label
16751,content_cf56e2e2e282,61678,review_for_refresh,stale_high_volume,61678,194,19.7,1
16514,content_7368877ea310,59472,review_for_refresh,stale_high_volume,59472,194,24.8,1
7021,content_1bfaa38ff26c,25715,review_for_refresh,stale_high_volume,25715,194,22.2,1
21268,content_0a91db491d14,13299,review_for_refresh,stale_high_volume,13299,193,10.5,1
11489,content_5feee3994adb,7812,review_for_refresh,stale_high_volume,7812,194,39.0,1
12045,content_c2d929d83eaa,7558,review_for_refresh,stale_high_volume,7558,193,17.9,1
698,content_b16bd7307b39,4590,review_for_refresh,stale_high_volume,4590,194,31.0,1
5327,content_fe16a55cd13d,4556,review_for_refresh,stale_high_volume,4556,194,16.4,1
26810,content_ecb6215e79fd,4429,review_for_refresh,stale_high_volume,4429,194,25.3,1
20837,content_928af3e22c80,1697,review_for_refresh,stale_high_volume,1697,193,15.8,1


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import numpy as np

# 1. Tính toán điểm số minh bạch (không dùng trọng số học được)[cite: 2]
# Lưu ý: avg_position = 0 có nghĩa là "không có dữ liệu"[cite: 3]
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 1000).astype(int)
valid_rank = (df["avg_position"] > 0).astype(int)

# Điểm số: Chỉ tính cho các trang thỏa mãn 3 điều kiện trên, xếp hạng ưu tiên theo impressions_90d
df["baseline_score"] = stale * visible * valid_rank * df["impressions_90d"]

# 2. Gán action và reason code[cite: 2]
df["reason_code"] = np.where(df["baseline_score"] > 0, "stale_high_volume", "none")
df["action"] = np.where(df["baseline_score"] > 0, "review_for_refresh", "ignore")

# 3. Tạo hàng đợi (ranked queue) bằng cách sắp xếp theo điểm số giảm dần
eval_df = df.dropna(subset=["baseline_score", "is_declining_label"]).copy()
output_cols = [
    "content_id", "baseline_score", "action", "reason_code",
    "impressions_90d", "days_since_last_update", "avg_position", "is_declining_label"
]
ranked_queue = eval_df.sort_values(by="baseline_score", ascending=False)[output_cols]

# 4. Đánh giá bằng độ đo Precision@K[cite: 2]
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Base rate (random picking): {eval_df['is_declining_label'].mean():.4f}")
print(f"Precision@10: {precision_at_k(eval_df['baseline_score'], eval_df['is_declining_label'], 10):.4f}")
print(f"Precision@50: {precision_at_k(eval_df['baseline_score'], eval_df['is_declining_label'], 50):.4f}")

# 5. Xuất file CSV[cite: 4]
# Sử dụng thư mục ../work/outputs/ vì notebook đang nằm trong work/notebooks/
os.makedirs("../work/outputs", exist_ok=True)
file_path = "../work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(file_path, index=False)

print(f"\nĐã ghi {len(ranked_queue)} dòng vào {file_path}")

Base rate (random picking): 0.6572
Precision@10: 1.0000
Precision@50: 0.7800

Đã ghi 30000 dòng vào ../work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
# Hiển thị 20 kết quả đứng đầu trong hàng đợi để tiến hành đánh giá thủ công
top_20 = ranked_queue.head(20)
display(top_20)

,content_id,baseline_score,action,reason_code,impressions_90d,days_since_last_update,avg_position,is_declining_label
16751,content_cf56e2e2e282,61678,review_for_refresh,stale_high_volume,61678,194,19.7,1
16514,content_7368877ea310,59472,review_for_refresh,stale_high_volume,59472,194,24.8,1
7021,content_1bfaa38ff26c,25715,review_for_refresh,stale_high_volume,25715,194,22.2,1
21268,content_0a91db491d14,13299,review_for_refresh,stale_high_volume,13299,193,10.5,1
11489,content_5feee3994adb,7812,review_for_refresh,stale_high_volume,7812,194,39.0,1
12045,content_c2d929d83eaa,7558,review_for_refresh,stale_high_volume,7558,193,17.9,1
698,content_b16bd7307b39,4590,review_for_refresh,stale_high_volume,4590,194,31.0,1
5327,content_fe16a55cd13d,4556,review_for_refresh,stale_high_volume,4556,194,16.4,1
26810,content_ecb6215e79fd,4429,review_for_refresh,stale_high_volume,4429,194,25.3,1
20837,content_928af3e22c80,1697,review_for_refresh,stale_high_volume,1697,193,15.8,1


**Đánh giá thủ công Top 20:**

* **Row 1-5:** Action: `review_for_refresh` | Reason: `stale_high_volume` | Confidence: High | **Wrong if:** Các trang này thuộc dạng nội dung "Evergreen" (ví dụ: định nghĩa toán học, khái niệm nền tảng). Nội dung này luôn đúng theo thời gian, nên việc không cập nhật hơn 180 ngày là bình thường, không cần thiết phải làm mới.
* **Row 6-10:** Action: `review_for_refresh` | Reason: `stale_high_volume` | Confidence: Medium | **Wrong if:** Sự suy giảm lưu lượng truy cập thực chất đến từ lỗi kỹ thuật đo lường (tracking bugs của GA4/GSC) hoặc do toàn bộ domain bị thuật toán phạt, chứ không phải do nội dung của bài viết bị lỗi thời.
* **Row 11-15:** Action: `review_for_refresh` | Reason: `stale_high_volume` | Confidence: Medium | **Wrong if:** Lượng hiển thị (`impressions_90d`) rất cao nhưng đến từ các từ khóa không mang lại giá trị chuyển đổi kinh doanh. Nếu làm mới các trang này, đội ngũ nội dung sẽ lãng phí tài nguyên mà không mang lại doanh thu.
* **Row 16-20:** Action: `review_for_refresh` | Reason: `stale_high_volume` | Confidence: Low | **Wrong if:** Vị trí xếp hạng trung bình (`avg_position`) của các trang này vốn đã quá thấp (ví dụ: hạng 60 - 80). Ở vị trí này, một đợt cập nhật nội dung đơn thuần hiếm khi đủ sức đẩy bài viết lên trang đầu tiên của kết quả tìm kiếm.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Danh sách các đặc trưng (features) đã sử dụng để tính điểm baseline
features_used = ["days_since_last_update", "impressions_90d", "avg_position"]

# Danh sách các cột chứa tín hiệu từ tương lai hoặc dẫn xuất trực tiếp ra nhãn (label trap)
leakage_cols = ["trend_pct", "trend_direction", "is_declining_label"]

print("--- KIỂM TRA RÒ RỈ DỮ LIỆU (LEAKAGE CHECK) ---")
has_leak = False
for col in leakage_cols:
    if col in features_used:
        print(f"❌ CẢNH BÁO: Rò rỉ dữ liệu! Cột '{col}' đang được sử dụng.")
        has_leak = True

if not has_leak:
    print("✅ Leakage check passed: Không có nhãn, cờ sản phẩm (product flags) hay tín hiệu từ cửa sổ tương lai (future windows) bị rò rỉ vào luật tính điểm.")

--- KIỂM TRA RÒ RỈ DỮ LIỆU (LEAKAGE CHECK) ---
✅ Leakage check passed: Không có nhãn, cờ sản phẩm (product flags) hay tín hiệu từ cửa sổ tương lai (future windows) bị rò rỉ vào luật tính điểm.


**Weak picks analysis:**
* **Những lựa chọn sai lầm (Weak picks):** Điểm yếu lớn nhất của bộ luật này là nó thiên vị các trang có lượng hiển thị (`impressions_90d`) khổng lồ do sử dụng phép nhân trực tiếp vào điểm số. Nếu một bài viết có hàng triệu lượt xem và thứ hạng đang cực kỳ ổn định ở vị trí Top 1 (không hề bị suy giảm), nó vẫn có thể lọt vào nhóm ưu tiên chỉ vì nó đã hơn 180 ngày chưa được chỉnh sửa. Điều này dẫn đến việc đề xuất làm mới những trang không thực sự cần thiết.

**Leakage confirmation:**
* **Xác nhận không rò rỉ:** Đã xác nhận không có cờ sản phẩm (product flags) hay dữ liệu từ cửa sổ tương lai (future windows) nào lọt vào mô hình.
* Đặc biệt, luật tính điểm tuyệt đối tuân thủ quy tắc không sử dụng `trend_pct` hay `trend_direction` làm feature, vì đây là những cột trực tiếp dẫn xuất ra nhãn `is_declining_label`[cite: 3].

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.